# CP4 — SHAP Explainability Analysis
**Thesis:** An Explainable ML Analysis of Race Outcome Determinants in the 2022–2025 F1 Ground-Effect Era.

Checkpoint 4 of 5. Explains the two **Random Forest** models selected in CP3 with **SHAP (TreeExplainer)**:
contrasts what is knowable *before* the race (Model A) against what *drove* the outcome *during* the race
(Model B). **No model export here — that is CP5.**

### Two models (both Random Forest, selected in CP3)
| | Model A — Pre-race (forecasting) | Model B — Full reconstruction |
|---|---|---|
| Features | 13 pre-race only | 24 incl. in-race pace/strategy/incidents |
| Test MAE | 3.332 | 2.545 |
| Test R² | 0.440 | 0.646 |
| Spearman | 0.643 | 0.803 |

The MAE gap (~0.79 positions) and Spearman gap (~0.16) quantify the **information value of race execution**
over starting conditions. SHAP of both models explains where that gap comes from.

### Decisions / deviations from the CP4 spec (verified against our CP2/CP3 outputs)
- **`pit_lane_start` does not exist.** The CP4 spec's `PRE_RACE_FEATURES` lists a `pit_lane_start` column,
  but in CP2 we instead **remapped `grid_position 0 → 21`** (pit-lane = worst slot, behind P20) so the
  pit-lane signal lives *inside* `grid_position` and stays monotonic for the trees. There is no separate
  `pit_lane_start` feature. We therefore load **both feature lists from disk**
  (`models/feature_list.json`, `models/pre_race_feature_list.json`) — the exact lists CP3 trained on —
  so the explained models match the trained models. Model A's 13th feature is `total_race_laps`, not
  `pit_lane_start`.
- **Models retrained from scratch** here with the CP3 hyperparameters/seeds and the same train-only median
  imputer; a sanity check confirms the reproduced test MAE matches CP3 (3.332 / 2.545) before any SHAP.
- **SHAP direction:** positive SHAP → pushes prediction toward a **higher** position number = **worse**
  finish; negative SHAP → toward a lower number = **better** finish. All plots are labelled accordingly.
- **Human-readable feature labels** are applied to every plot.

## Setup

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
FIG_DIR = ROOT / "reports" / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
                     "axes.titleweight": "bold"})
saved_figures = []
def save_cur(name):
    """Save the current matplotlib figure (SHAP plots draw on the active figure)."""
    plt.tight_layout()
    plt.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")
    plt.close()
    saved_figures.append(name)
    print("  saved ->", f"reports/figures/{name}")

# Feature lists exactly as CP3 trained them (source of truth on disk).
FULL_FEATURES = json.load(open(MODELS_DIR / "feature_list.json"))            # Model B (24)
PRE_RACE_FEATURES = json.load(open(MODELS_DIR / "pre_race_feature_list.json"))  # Model A (13)
TARGET = "final_position"

# Human-readable labels for every feature (thesis-quality axes).
FEATURE_LABELS = {
    "grid_position": "Grid Position",
    "driver_rolling_avg_3": "Driver Form (3-race avg)",
    "driver_rolling_avg_5": "Driver Form (5-race avg)",
    "driver_dnf_rate_season": "Driver DNF Rate (season)",
    "driver_encoded": "Driver Identity",
    "constructor_rolling_avg_points_3": "Constructor Form (3-race avg)",
    "constructor_season_points": "Constructor Season Points",
    "team_encoded": "Constructor Identity",
    "num_pit_stops": "Number of Pit Stops",
    "compound_encoded": "Starting Tyre Compound",
    "is_wet_race": "Wet Race",
    "safety_car_deployed": "Safety Car Deployed",
    "vsc_deployed": "Virtual Safety Car",
    "red_flag": "Red Flag",
    "is_street_circuit": "Street Circuit",
    "total_race_laps": "Race Length (laps)",
    "season": "Season",
    "avg_air_temp": "Air Temperature",
    "avg_track_temp": "Track Temperature",
    "median_lap_time": "Median Lap Time",
    "pace_delta": "Pace vs Field Median",
    "clean_pace_delta": "Clean Air Pace Delta",
    "teammate_pace_delta": "Pace vs Teammate",
    "first_stint_len": "First Stint Length",
    "pit_lane_start": "Pit Lane Start",
}
labels_a = [FEATURE_LABELS.get(f, f) for f in PRE_RACE_FEATURES]
labels_b = [FEATURE_LABELS.get(f, f) for f in FULL_FEATURES]
print("Model B full features :", len(FULL_FEATURES))
print("Model A pre-race feats:", len(PRE_RACE_FEATURES))
print("shap", shap.__version__)

Model B full features : 24
Model A pre-race feats: 13
shap 0.52.0


## Load data & replicate CP3 preprocessing
Identical known-issue handling to CP3 so the retrained models match exactly: unseen rookie driver/team
codes → median **train** code; `constructor_season_points` NaN (season start) → 0. Median imputation is
applied after (fit on train only).

In [2]:
# Model B (full) frames
train_b = pd.read_csv(PROC / "train.csv").reset_index(drop=True)
test_b  = pd.read_csv(PROC / "test.csv").reset_index(drop=True)
# Model A (pre-race) frames
train_a = pd.read_csv(PROC / "pre_race" / "pre_race_train.csv").reset_index(drop=True)
test_a  = pd.read_csv(PROC / "pre_race" / "pre_race_test.csv").reset_index(drop=True)

def remap_rookies(train_df, test_df, id_col):
    """Unseen codes in test -> median train code (CP3 rule). Returns rookie abbrevs for reporting."""
    known = set(train_df[id_col].unique())
    med = int(np.round(train_df[id_col].median()))
    mask = ~test_df[id_col].isin(known)
    rookies = sorted(test_df.loc[mask, id_col].unique())
    test_df[id_col] = test_df[id_col].where(test_df[id_col].isin(known), med)
    return rookies, med

# Same remap for both models (driver_encoded / team_encoded come from the same CP2 encoders).
abbrev_col_b = "driver"        # full test.csv carries the 3-letter abbrev as 'driver'
abbrev_col_a = "driver_abbrev" # pre-race files carry it as 'driver_abbrev'
for tr, te in [(train_b, test_b), (train_a, test_a)]:
    remap_rookies(tr, te, "driver_encoded")
    remap_rookies(tr, te, "team_encoded")
    for d in (tr, te):
        d["constructor_season_points"] = d["constructor_season_points"].fillna(0.0)

rookie_codes = sorted(set(pd.read_csv(PROC / "test.csv")["driver_encoded"].unique())
                      - set(train_b["driver_encoded"].unique()))
rookie_abbr = sorted(pd.read_csv(PROC / "test.csv")
                     .loc[lambda d: ~d["driver_encoded"].isin(train_b["driver_encoded"].unique()), "driver"].unique())
print("Rookie driver codes remapped:", rookie_codes, rookie_abbr)

X_train_a, y_train_a = train_a[PRE_RACE_FEATURES].copy(), train_a[TARGET].values
X_test_a,  y_test_a  = test_a[PRE_RACE_FEATURES].copy(),  test_a[TARGET].values
X_train_b, y_train_b = train_b[FULL_FEATURES].copy(),     train_b[TARGET].values
X_test_b,  y_test_b  = test_b[FULL_FEATURES].copy(),      test_b[TARGET].values
print("A:", X_train_a.shape, X_test_a.shape, "| B:", X_train_b.shape, X_test_b.shape)

Rookie driver codes remapped: [np.int64(2), np.int64(4), np.int64(10)] ['ANT', 'BOR', 'HAD']
A: (1350, 13) (476, 13) | B: (1350, 24) (476, 24)


## Retrain both Random Forest models from scratch
Same hyperparameters and seed as CP3. Median imputer fit on **train only**, then both matrices are
transformed (kept as DataFrames so SHAP sees feature names). The printed test MAEs must match CP3
(Model A ≈ 3.332, Model B ≈ 2.545) — a guard that the explained model = the trained model.

In [3]:
rf_params = dict(n_estimators=500, max_depth=None, min_samples_leaf=2,
                 max_features="sqrt", random_state=42, n_jobs=-1)

# Model A
imp_a = SimpleImputer(strategy="median").fit(X_train_a)
Xtr_a = pd.DataFrame(imp_a.transform(X_train_a), columns=PRE_RACE_FEATURES)
Xte_a = pd.DataFrame(imp_a.transform(X_test_a),  columns=PRE_RACE_FEATURES)
model_a = RandomForestRegressor(**rf_params).fit(Xtr_a, y_train_a)

# Model B
imp_b = SimpleImputer(strategy="median").fit(X_train_b)
Xtr_b = pd.DataFrame(imp_b.transform(X_train_b), columns=FULL_FEATURES)
Xte_b = pd.DataFrame(imp_b.transform(X_test_b),  columns=FULL_FEATURES)
model_b = RandomForestRegressor(**rf_params).fit(Xtr_b, y_train_b)

mae_a = mean_absolute_error(y_test_a, model_a.predict(Xte_a))
mae_b = mean_absolute_error(y_test_b, model_b.predict(Xte_b))
print(f"Model A (pre-race) test MAE: {mae_a:.3f}  (CP3: 3.332)")
print(f"Model B (full)     test MAE: {mae_b:.3f}  (CP3: 2.545)")

Model A (pre-race) test MAE: 3.332  (CP3: 3.332)
Model B (full)     test MAE: 2.545  (CP3: 2.545)


## SHAP — TreeExplainer
`TreeExplainer` is exact and fast for Random Forest. SHAP values are computed on the **2025 test set**
only and are in units of finishing positions.

In [4]:
explainer_a = shap.TreeExplainer(model_a)
explainer_b = shap.TreeExplainer(model_b)

# Explanation objects (carry base value + readable feature names for waterfall/plots).
exp_a = explainer_a(Xte_a); exp_a.feature_names = labels_a
exp_b = explainer_b(Xte_b); exp_b.feature_names = labels_b
shap_values_a = exp_a.values   # (476, 13)
shap_values_b = exp_b.values   # (476, 24)
print("SHAP A:", shap_values_a.shape, "| SHAP B:", shap_values_b.shape)
print("Direction: +SHAP -> higher position number (worse finish); -SHAP -> better finish.")

# Labelled copies of the test matrices for plotting.
Xte_a_lab = Xte_a.copy(); Xte_a_lab.columns = labels_a
Xte_b_lab = Xte_b.copy(); Xte_b_lab.columns = labels_b

SHAP A: (476, 13) | SHAP B: (476, 24)
Direction: +SHAP -> higher position number (worse finish); -SHAP -> better finish.


## Model A (Pre-race) — SHAP plots  ·  prefix `15_`

In [5]:
# Plot 15a — Beeswarm summary (Model A)
shap.summary_plot(shap_values_a, Xte_a, feature_names=labels_a, show=False)
plt.title("Model A (Pre-race): Feature Impact on Finishing Position\n(+ = worse finish, - = better)")
save_cur("15a_shap_beeswarm_prerace.png"); plt.show()

  saved -> reports/figures/15a_shap_beeswarm_prerace.png


In [6]:
# Plot 15b — Mean |SHAP| bar (Model A)
shap.summary_plot(shap_values_a, Xte_a, feature_names=labels_a, plot_type="bar", show=False)
plt.title("Model A (Pre-race): Mean Feature Importance")
plt.xlabel("Mean |SHAP| (positions)")
save_cur("15b_shap_bar_prerace.png"); plt.show()

  saved -> reports/figures/15b_shap_bar_prerace.png


In [7]:
# Plots 15c/d/e — Dependence plots for the top-3 Model A features by mean |SHAP|
mean_abs_a = np.abs(shap_values_a).mean(axis=0)
top3_a = np.argsort(mean_abs_a)[::-1][:3]
for rank, fi in enumerate(top3_a):
    orig, lab = PRE_RACE_FEATURES[fi], labels_a[fi]
    shap.dependence_plot(lab, shap_values_a, Xte_a_lab, show=False)
    plt.title(f"Model A Dependence: {lab}  (+SHAP = worse finish)")
    save_cur(f"15{'cde'[rank]}_shap_dependence_prerace_{orig}.png"); plt.show()

  saved -> reports/figures/15c_shap_dependence_prerace_grid_position.png
  saved -> reports/figures/15d_shap_dependence_prerace_constructor_rolling_avg_points_3.png
  saved -> reports/figures/15e_shap_dependence_prerace_driver_rolling_avg_5.png


## Model B (Full reconstruction) — SHAP plots  ·  prefix `16_`

In [8]:
# Plot 16a — Beeswarm summary (Model B)
shap.summary_plot(shap_values_b, Xte_b, feature_names=labels_b, show=False)
plt.title("Model B (Full Reconstruction): Feature Impact on Finishing Position\n(+ = worse finish, - = better)")
save_cur("16a_shap_beeswarm_full.png"); plt.show()

  saved -> reports/figures/16a_shap_beeswarm_full.png


In [9]:
# Plot 16b — Mean |SHAP| bar (Model B)
shap.summary_plot(shap_values_b, Xte_b, feature_names=labels_b, plot_type="bar", show=False)
plt.title("Model B (Full Reconstruction): Mean Feature Importance")
plt.xlabel("Mean |SHAP| (positions)")
save_cur("16b_shap_bar_full.png"); plt.show()

  saved -> reports/figures/16b_shap_bar_full.png


In [10]:
# Plots 16c/d/e — Dependence plots for the top-3 Model B features by mean |SHAP|
mean_abs_b = np.abs(shap_values_b).mean(axis=0)
top3_b = np.argsort(mean_abs_b)[::-1][:3]
for rank, fi in enumerate(top3_b):
    orig, lab = FULL_FEATURES[fi], labels_b[fi]
    shap.dependence_plot(lab, shap_values_b, Xte_b_lab, show=False)
    plt.title(f"Model B Dependence: {lab}  (+SHAP = worse finish)")
    save_cur(f"16{'cde'[rank]}_shap_dependence_full_{orig}.png"); plt.show()

  saved -> reports/figures/16c_shap_dependence_full_pace_delta.png
  saved -> reports/figures/16d_shap_dependence_full_clean_pace_delta.png
  saved -> reports/figures/16e_shap_dependence_full_grid_position.png


## Plot 17 — Feature importance: Pre-race vs Full reconstruction
The key thesis plot. Both panels are mean |SHAP| (positions). In Model B, **red bars are in-race-only
features** (present in B, absent from A) — these are the signal that race execution adds on top of the
pre-race picture.

In [11]:
mean_a = np.abs(shap_values_a).mean(0)
mean_b = np.abs(shap_values_b).mean(0)
dfa = (pd.DataFrame({"feat": PRE_RACE_FEATURES, "imp": mean_a})
       .assign(label=lambda d: d["feat"].map(lambda f: FEATURE_LABELS.get(f, f)))
       .sort_values("imp"))
dfb = (pd.DataFrame({"feat": FULL_FEATURES, "imp": mean_b})
       .assign(shared=lambda d: d["feat"].isin(PRE_RACE_FEATURES),
               label=lambda d: d["feat"].map(lambda f: FEATURE_LABELS.get(f, f)))
       .sort_values("imp"))

fig, axes = plt.subplots(1, 2, figsize=(15, 8))
axes[0].barh(dfa["label"], dfa["imp"], color="#1f77b4")
axes[0].set_title("Model A (Pre-race) — 13 features")
axes[0].set_xlabel("Mean |SHAP| (positions)")
axes[1].barh(dfb["label"], dfb["imp"],
             color=["#1f77b4" if s else "#d62728" for s in dfb["shared"]])
axes[1].set_title("Model B (Full) — red = in-race-only signal")
axes[1].set_xlabel("Mean |SHAP| (positions)")
fig.suptitle("Feature Importance: Pre-race vs Full Reconstruction", fontsize=14, fontweight="bold")
save_cur("17_shap_comparison_a_vs_b.png"); plt.show()

  saved -> reports/figures/17_shap_comparison_a_vs_b.png


## Plot 18 — Waterfall: the single hardest-to-predict driver-race (Model B)
Selection: the 2025 race with the highest mean per-race MAE, then the driver in that race with the
largest absolute prediction error. The waterfall decomposes that one prediction into per-feature SHAP
contributions from the model base value.

In [12]:
tb = test_b.copy()
tb["pred_b"] = model_b.predict(Xte_b)
tb["error"]  = np.abs(tb["pred_b"] - tb[TARGET])

per_race = tb.groupby(["round", "event_name"])["error"].mean()
worst_round, worst_race_name = per_race.idxmax()
wr = tb[tb["round"] == worst_round]
worst_pos = int(wr["error"].idxmax())          # positional index (tb was reset_index)
worst_driver = tb.loc[worst_pos, "driver"]
actual_pos = tb.loc[worst_pos, TARGET]
pred_pos   = tb.loc[worst_pos, "pred_b"]
print(f"Worst race  : {worst_race_name} (round {worst_round}), mean MAE {per_race.max():.2f}")
print(f"Worst driver: {worst_driver}  | Actual P{actual_pos:.0f}  Predicted P{pred_pos:.1f}"
      f"  Error {abs(actual_pos - pred_pos):.1f}")

# Top-2 SHAP drivers for this specific prediction
row_sv = shap_values_b[worst_pos]
top2_idx = np.argsort(np.abs(row_sv))[::-1][:2]
top2 = [(FEATURE_LABELS.get(FULL_FEATURES[i], FULL_FEATURES[i]), float(row_sv[i])) for i in top2_idx]
print("Top-2 SHAP drivers:", top2)

shap.waterfall_plot(exp_b[worst_pos], show=False)
plt.title(f"SHAP Waterfall: {worst_driver} at {worst_race_name}\n"
          f"Predicted P{pred_pos:.1f} | Actual P{actual_pos:.0f}  (+ = worse finish)")
save_cur("18_shap_waterfall.png"); plt.show()

Worst race  : Dutch Grand Prix (round 15), mean MAE 4.50
Worst driver: NOR  | Actual P18  Predicted P3.2  Error 14.8
Top-2 SHAP drivers: [('Pace vs Field Median', -2.3045318926936487), ('Clean Air Pace Delta', -1.9055538598775918)]
  saved -> reports/figures/18_shap_waterfall.png


## Key SHAP findings

In [13]:
def ranked(shap_vals, feats):
    m = np.abs(shap_vals).mean(0)
    return [(feats[i], float(m[i])) for i in np.argsort(m)[::-1]]

ra, rb = ranked(shap_values_a, PRE_RACE_FEATURES), ranked(shap_values_b, FULL_FEATURES)
rank_a = {f: i + 1 for i, (f, _) in enumerate(ra)}
rank_b = {f: i + 1 for i, (f, _) in enumerate(rb)}

print("=== SHAP KEY FINDINGS ===")
print("\n--- Model A (Pre-race): top 5 by mean |SHAP| ---")
for f, v in ra[:5]:
    print(f"  {FEATURE_LABELS.get(f, f):32s} {v:.3f} positions")
print("\n--- Model B (Full): top 5 by mean |SHAP| ---")
for f, v in rb[:5]:
    print(f"  {FEATURE_LABELS.get(f, f):32s} {v:.3f} positions")

print("\n--- The gap: in-race-only features (in B, not in A) ---")
in_race_only = [f for f in FULL_FEATURES if f not in PRE_RACE_FEATURES]
imp_b_map = dict(rb)
for f, v in sorted(((f, imp_b_map[f]) for f in in_race_only), key=lambda x: -x[1]):
    print(f"  {FEATURE_LABELS.get(f, f):32s} {v:.3f}  (Model B rank {rank_b[f]})")

print("\n--- Notable ---")
gi = FULL_FEATURES.index("grid_position")
sp_grid = spearmanr(Xte_b["grid_position"], shap_values_b[:, gi]).correlation
print(f"  Spearman(grid_position value, its SHAP) = {sp_grid:.3f} "
      f"(positive => starting further back pushes toward a worse finish, as expected)")
print(f"  driver_encoded rank -> Model A: {rank_a['driver_encoded']}, Model B: {rank_b['driver_encoded']}")
print(f"  pace_delta rank (B): {rank_b['pace_delta']} | teammate_pace_delta rank (B): {rank_b['teammate_pace_delta']}"
      f" | clean_pace_delta rank (B): {rank_b['clean_pace_delta']}")

# Caveats triggered by the data
if rank_a["driver_encoded"] <= 5 or rank_b["driver_encoded"] <= 5:
    print("\n  [CAVEAT] driver_encoded is a label integer; its SHAP weight partly reflects memorisation of")
    print("           driver-team pairings, not a generalisable skill signal. The interpretable driver-skill")
    print("           axis is the rolling-form features (driver_rolling_avg_3/5).")
if rank_a.get("is_wet_race", 99) <= 5:
    print("  [CAVEAT] is_wet_race ranks high in Model A but assumes weather-forecast accuracy (pre-race proxy).")

=== SHAP KEY FINDINGS ===

--- Model A (Pre-race): top 5 by mean |SHAP| ---
  Grid Position                    1.547 positions
  Constructor Form (3-race avg)    0.947 positions
  Driver Form (5-race avg)         0.734 positions
  Constructor Season Points        0.451 positions
  Driver Form (3-race avg)         0.341 positions

--- Model B (Full): top 5 by mean |SHAP| ---
  Pace vs Field Median             1.601 positions
  Clean Air Pace Delta             1.175 positions
  Grid Position                    0.578 positions
  Driver Form (5-race avg)         0.418 positions
  Pace vs Teammate                 0.382 positions

--- The gap: in-race-only features (in B, not in A) ---
  Pace vs Field Median             1.601  (Model B rank 1)
  Clean Air Pace Delta             1.175  (Model B rank 2)
  Pace vs Teammate                 0.382  (Model B rank 5)
  Number of Pit Stops              0.244  (Model B rank 7)
  First Stint Length               0.201  (Model B rank 10)
  Safety Car De

## CP4 Summary

In [14]:
print("=== CP4 COMPLETE ===")
print("Model A (Pre-race) SHAP:")
for i, (f, v) in enumerate(ra[:3]):
    print(f"  {['Top feature','2nd feature','3rd feature'][i]:16s}: {FEATURE_LABELS.get(f, f)} "
          f"(mean |SHAP| = {v:.2f} positions)")
print(f"  driver_encoded rank: {rank_a['driver_encoded']}"
      + ("  [TOP 5 -> memorisation caveat]" if rank_a['driver_encoded'] <= 5 else ""))

print("\nModel B (Full) SHAP:")
for i, (f, v) in enumerate(rb[:3]):
    print(f"  {['Top feature','2nd feature','3rd feature'][i]:16s}: {FEATURE_LABELS.get(f, f)} "
          f"(mean |SHAP| = {v:.2f} positions)")
print(f"  driver_encoded rank: {rank_b['driver_encoded']}"
      + ("  [TOP 5 -> memorisation caveat]" if rank_b['driver_encoded'] <= 5 else ""))

print("\nIn-race features SHAP contribution (Model B only):")
for f, v in sorted(((f, imp_b_map[f]) for f in in_race_only), key=lambda x: -x[1])[:4]:
    print(f"  {FEATURE_LABELS.get(f, f)}: {v:.2f} positions")

print("\nWaterfall plot:")
print(f"  Race  : {worst_race_name}")
print(f"  Driver: {worst_driver}")
print(f"  Actual: P{actual_pos:.0f} | Predicted: P{pred_pos:.1f} | Error: {abs(actual_pos - pred_pos):.1f} positions")
print(f"  Reason: {top2[0][0]} (SHAP {top2[0][1]:+.2f}), {top2[1][0]} (SHAP {top2[1][1]:+.2f})")

print(f"\nPlots saved: {len(saved_figures)} files in reports/figures/")
print("  15a/15b/15c-e — Model A beeswarm, bar, dependence")
print("  16a/16b/16c-e — Model B beeswarm, bar, dependence")
print("  17            — A vs B importance comparison")
print("  18            — Waterfall (hardest single prediction)")
print("Known issues for CP5:")
print("  - driver_encoded / team_encoded are label codes: high SHAP = partly memorisation, not skill.")
print("  - is_wet_race in Model A assumes forecast accuracy (pre-race proxy).")
print("  - Models retrained in-notebook (not loaded); CP5 will persist the selected RF + imputer.")

=== CP4 COMPLETE ===
Model A (Pre-race) SHAP:
  Top feature     : Grid Position (mean |SHAP| = 1.55 positions)
  2nd feature     : Constructor Form (3-race avg) (mean |SHAP| = 0.95 positions)
  3rd feature     : Driver Form (5-race avg) (mean |SHAP| = 0.73 positions)
  driver_encoded rank: 10

Model B (Full) SHAP:
  Top feature     : Pace vs Field Median (mean |SHAP| = 1.60 positions)
  2nd feature     : Clean Air Pace Delta (mean |SHAP| = 1.18 positions)
  3rd feature     : Grid Position (mean |SHAP| = 0.58 positions)
  driver_encoded rank: 15

In-race features SHAP contribution (Model B only):
  Pace vs Field Median: 1.60 positions
  Clean Air Pace Delta: 1.18 positions
  Pace vs Teammate: 0.38 positions
  Number of Pit Stops: 0.24 positions

Waterfall plot:
  Race  : Dutch Grand Prix
  Driver: NOR
  Actual: P18 | Predicted: P3.2 | Error: 14.8 positions
  Reason: Pace vs Field Median (SHAP -2.30), Clean Air Pace Delta (SHAP -1.91)

Plots saved: 12 files in reports/figures/
  15a/15b/